<!-- # An Image is worth 16x16 words: Transformers for Image Recognition at Scale -->
# 一张图像相当于16x16个词：大规模图像识别的变换器

- [论文](https://arxiv.org/abs/2010.11929)
- [代码](https://github.com/google-research/vision_transformer)

<a id="model_arch" />
<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/model_scheme.png" />
    <span style="color: black; font-size: 12px;">图 1: 模型架构</span>
</div>

## 方法论 Method

- 模型概述见[图 1](#model_arch)。标准Transformer的输入是一维令牌token嵌入序列。为了处理二维图像，
  1. 我们将图像`image` $x \in \mathbb{R}^{H\times W\times C}$ 
  2. 重塑为一系列二维图像块`patches` $x_p \in \mathbb{R}^{N\times (P^2\cdot C)}$，
  3. 其中$(H,W)$是原始图像的分辨率， $C$是通道数， $(P,P)$是每个图像块的分辨率， $N=\frac{HW}{P^2}$分块数，这也是变换器的有效输入序列长度。
  4. 变换器在所有层中使用恒定的潜在向量维度$D$，因此我们将图像块展平后，并通过相同的线性投影层$E$将输入映射到$D$维嵌入[等式 1](#equations)。我们称该投影的输出为图像块嵌入。
     - 这一步和使用$D$个大小为$P\times P \times C$，步幅为$P$的卷积操作是等价的。



- 类似BERT的`[class]`token,我们在patch嵌入序列前添加一个可以学习的嵌入$z_0^0 = x_\text{class}$(size:$D$)，其在Transformer encoder的输出$z_L^0$作为图像的表示$y$[等式 4](#equations)。
  - $z_l^i$表示第$i$个位置在第$l$层的Transformer编码器输出。($l=0$表示输入嵌入，$l=L$表示最后一层输出)
- 在预训练和微调过程中，都会在$z_L^0$上附加一个分类头。预训练时，分类头由一个一层隐藏层的MLP实现，微调时由一个单线性层实现。

- 位置嵌入被添加到图像块嵌入中，以保留位置信息。我们使用化标准的可学习的1D位置嵌入，因为我们没有观察到使用更高级的2D感知位置嵌入有显著的性能提升。生成的嵌入向量序列作为编码器的输入。

- `Transformer`编码器

$$\begin{align}
    \mathbf{z}_0 &= [ \mathbf{x}_\text{class}; \, \mathbf{x}^1_p \mathbf{E}; \, \mathbf{x}^2_p \mathbf{E}; \cdots; \, \mathbf{x}^{N}_p \mathbf{E} ] + \mathbf{E}_{pos},
    && \mathbf{E} \in \mathbb{R}^{(P^2 \cdot C) \times D},\, \mathbf{E}_{pos}  \in \mathbb{R}^{(N + 1) \times D} \\
    \mathbf{z^\prime}_\ell &= \operatorname{MSA}(\operatorname{LN}(\mathbf{z}_{\ell-1})) + \mathbf{z}_{\ell-1}, && \ell=1\ldots L \\
    \mathbf{z}_\ell &= \operatorname{MLP}(\operatorname{LN}(\mathbf{z^\prime}_{\ell})) + \mathbf{z^\prime}_{\ell}, && \ell=1\ldots L \\
    \mathbf{y} &= \operatorname{LN}(\mathbf{z}_L^0)
\end{align}$$

<a id="equations" />

### 3.1 Vision Transformer

__归纳偏差 Inductive bias__
- 视觉 Transformer 比卷积神经网络具有更少的图像特定归纳偏差。
  - __图像特定的归纳偏差 Image-specific inductive bias__: 
      - 局部性: 图像中相邻像素通常比距离较远的像素更相关。
      - 平移不变性: 图像中的特征/对象无论出现在哪个位置，模型都能以上述方式识别。
      - 空间层次结构（Spatial Hierarchy）：CNN通常具有多层结构，随着网络层数的增加，特征图的空间分辨率逐渐降低，而通道数逐渐增加。这种层次/金字塔结构能让模型从不同尺度和层次上提取图像特征。（前期更关注局部细节，后期更关注全局语义）
  - **MLP**层是局部和平移不变的
  - **自注意力层**是全局的

__混合架构 Hybrid Architecture__
- 可以用卷积神经网络的特征图作为原始图像块的代替输入($x_p$)。
- 输入patch的空间大小可以是`1x1`，相当于扁平化特征图的空间维度。

### 3,2 微调和更高分辨率 Fine-tuning and higher resolutions

1. 移除了预训练的预测头($D \times 1000$, ImageNet)，并附加了一个初始化为零的$D \times K$前馈层，其中 $K$ 是下游类别的数量。
2. 修改了位置编码：
   - 预训练使用更高分辨率的图像。保持块大小不变，这会导致更大的有效序列长度。
   - 由于长度变化，预训练的位置嵌入可能不再有意义。因此，作者根据它们的原始位置进行二维插值。
   - 具体实现看[补充小节](#补充)中`interpolate_pos_encoding`函数。
   - 这是唯一一个将关于图像二维结构的归纳偏置手动注入Vision Transformer的地方。

## 4 实验 Experiments

### 4.1 实验设置

<a id="table1" />
<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/table1_vit_details.png" />
    <span style="color: black; font-size: 12px;"><strong>表1</strong>: ViT模型参数细节</span>
</div>

### 4.2 与SOTA的比较

<a id="table2" />
<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 60%; margin: auto;">
    <image src="./assets/table2_comparison.png" />
    <span style="color: black; font-size: 12px;"><strong>表2</strong>: 模型性能比较`</span>
</div>

### 4.5 分析视觉Transformer

<div style="background-color: white; padding: 10px; border-radius: 5px; text-align: center; width: 30%; margin: auto;">
    <image src="./assets/20201002_selected_attention_examples.png" />
    <span style="color: black; font-size: 12px;"><strong>图 6:</strong>输出token到输入空间的注意力典型示例。详见附录 D.7。</span>
</div>

## 补充

- `transformers/model/vit/modeling_vit.py`中ViT实现
```python
class ViTEmbeddings(nn.Module):
    def __init__(self, config: ViTConfig, use_mask_token: bool = False):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.hidden_size))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, config.hidden_size)) if use_mask_token else None
        self.patch_embeddings = ViTPatchEmbeddings(config)
        num_patches = self.patch_embeddings.num_patches
        self.position_embeddings = nn.Parameter(torch.randn(1, num_patches + 1, config.hidden_size))
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.patch_size = config.patch_size
        self.config = config

    def forward(
        self,
        pixel_values: torch.Tensor,
        bool_masked_pos: Optional[torch.BoolTensor] = None,
        interpolate_pos_encoding: bool = False,
    ) -> torch.Tensor:
        batch_size, num_channels, height, width = pixel_values.shape
        embeddings = self.patch_embeddings(pixel_values, interpolate_pos_encoding=interpolate_pos_encoding)
        if bool_masked_pos is not None:
            seq_length = embeddings.shape[1]
            mask_tokens = self.mask_token.expand(batch_size, seq_length, -1)
            # replace the masked visual tokens by mask_tokens
            mask = bool_masked_pos.unsqueeze(-1).type_as(mask_tokens)
            embeddings = embeddings * (1.0 - mask) + mask_tokens * mask
        # add the [CLS] token to the embedded patch tokens
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        embeddings = torch.cat((cls_tokens, embeddings), dim=1)
        # add positional encoding to each token
        if interpolate_pos_encoding:
            embeddings = embeddings + self.interpolate_pos_encoding(embeddings, height, width)
        else:
            embeddings = embeddings + self.position_embeddings
        embeddings = self.dropout(embeddings)
        return embeddings
```

### 位置编码

- `transformers`库中的`ViT`直接使用可学习的位置编码:`nn.Parameter(torch.randn(1, num_patches + 1, config.hidden_size))`
- 通过`interpolate_pos_encoding`方法将训练好的位置编码插入到不同分辨率下。



```python
# init函数中定义的position_embeddings
self.position_embeddings = nn.Parameter(torch.randn(1, num_patches + 1, config.hidden_size))
# 第二种
def interpolate_pos_encoding(self, embeddings: torch.Tensor, # [batch_size, num_patches+1, dim]
    height: int, width: int) -> torch.Tensor:
    """
    将预训练好的位置嵌入插值到新的图像分辨率。
    """
    num_patches = embeddings.shape[1] - 1
    num_positions = self.position_embeddings.shape[1] - 1
    # always interpolate when tracing to ensure the exported model works for dynamic input shapes
    if not torch.jit.is_tracing() and num_patches == num_positions and height == width:
        return self.position_embeddings
    class_pos_embed = self.position_embeddings[:, :1]
    patch_pos_embed = self.position_embeddings[:, 1:]
    dim = embeddings.shape[-1]
    new_height = height // self.patch_size
    new_width = width // self.patch_size
    sqrt_num_positions = torch_int(num_positions**0.5)
    patch_pos_embed = patch_pos_embed.reshape(1, sqrt_num_positions, sqrt_num_positions, dim)
    patch_pos_embed = patch_pos_embed.permute(0, 3, 1, 2)
    patch_pos_embed = nn.functional.interpolate(
        patch_pos_embed,
        size=(new_height, new_width),
        mode="bicubic",  # NT: 双三次插值，4x4邻域像素通过三次多项式插值计算新像素值
        align_corners=False,
    )
    patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)
    return torch.cat((class_pos_embed, patch_pos_embed), dim=1)
```

## 相关面试问题

__1. ViT为什么要分块图像？__
- 分块图像可以将二维图像转换为一维序列(2d => 1d)，适应Transformer的输入格式。
- 分块之后，图像变成固定大小的图像块，适配各种分辨率的图像输入，类似NLP中不同长度的文本序列。
- 分块之后，大小固定的块通过线性层/卷积层映射到潜在空间，减少了参数数量，提高了计算效率。注意力计算也从$O(N^2)$降低到$O((\frac{N}{P})^2)$。
- 分块之后可以更好适配位置嵌入。


__2. ViT中的位置编码:__


- ViT原文中使用的是绝对位置编码（APE），即使用一个可学习的位置嵌入矩阵。
- 之后的ViT变体中也尝试了多种不同的位置编码。
    - 比如CPVT中使用条件位置编码，通过卷积/池化动态生成位置向量，具有更好的平移不变性和分辨率泛化能力。

